# 🦁 تدريب Lion-1 (LoRA) — خط كامل على Colab

هذا الملف ينفّذ المسار كاملاً: **بيانات ← تدريب LoRA ← دمج ← GGUF كمّي**، وينتج ملفاً
تشغّله داخل التطبيق على هاتفك بلا إنترنت أو مزوّد خلفي.

## ما الذي يتعلّمه النموذج فعلاً
- **الأسلوب والهوية**: «أسد / Lion-1 من Star Syria»، عربية موجزة تبدأ بالجواب.
- **صيغة أدوات التطبيق**: `[[TOOL]]{"name":"file_read","args":{...}}[[/TOOL]]` بأسماء
  الأدوات ومفاتيح وسائطها الحقيقية (تُقرأ من `AiTools.kt` مباشرة عبر `data/import_tools.py`).
- **متى لا يستدعي أداة** (أسئلة معرفية) ومتى يسأل قبل عملية خطرة.

## ما لا يتعلّمه
المعرفة الموسوعية. نموذج 1.5B لا يحفظ موسوعات — المعرفة تأتي من أدوات التطبيق
(البحث/الجلب/الفهرس المحلي) لا من الأوزان.

## المتطلبات
- حساب Google + **GPU T4** (مجاني): `Runtime → Change runtime type → T4 GPU`.
- الزمن المتوقع: **45–90 دقيقة** للنموذج 1.5B، ومساحة ~8GB.

> ⚠️ استخدم نسخة من المستودع على حسابك (Fork) أو ارفع مجلد `model/` بنفسك إن لم يكن المستودع عاماً.

## ⚠️ أهم نقطة: أنت **لا ترفع** ملف LoRA

ملف LoRA هو **ناتج** هذا الدفتر لا مدخله. لا يوجد شيء ترفعه هنا سوى الدفتر نفسه (`.ipynb`).
ما تحتاجه فقط: حساب Google + GPU مجاني. بقية الملفات (الكود، البيانات) تُنزَّل تلقائياً من GitHub.

| ما ترفعه أنت | ما يصنعه الكولاب |
|---|---|
| الدفتر `Lion1_LoRA_Colab.ipynb` | البيانات + تدريب LoRA + الدمج + ملف GGUF النهائي |

الدليل المصوّر خطوة بخطوة (بالنقرات): `docs/LION1_COLAB_STEPBYSTEP_AR.md`


In [ ]:
#@title ١) جلب كود المشروع (لا ترفع شيئاً — الدفتر ينزّله)
import os, getpass, subprocess
REPO = "https://github.com/starmaen/LionAI.git"   # ضع رابط fork إن كان المستودع خاصاً بك
TOKEN = ""  # اتركه فارغاً للمستودع العام. للمستودع الخاص: توكن GitHub بصلاحية repo
if not os.path.isdir("/content/LionAI"):
    url = REPO
    if TOKEN:
        url = REPO.replace("https://", f"https://x-access-token:{TOKEN}@")
    r = subprocess.run(["git", "clone", "--depth", "1", url, "/content/LionAI"])
    if r.returncode != 0:
        raise SystemExit("تعذّر جلب المستودع: إن كان خاصاً ضع التوكن في TOKEN أعلاه، أو اعمل Fork عاماً.")
%cd /content/LionAI/model
!ls


In [ ]:
#@title ٢) تثبيت المكتبات (٣–٥ دقائق)
!pip install -q --upgrade "transformers>=4.46" "peft>=0.13" "trl>=0.12" datasets accelerate bitsandbytes sentencepiece
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "لا يوجد — بدّل إلى T4 من Runtime")


In [ ]:
#@title ٣) توليد البيانات والتحقق منها (بلا شبكة)
# يستورد كتالوج الأدوات الحقيقي من كود التطبيق، ثم يولّد train/val، ثم يفحص الصيغة.
!python data/import_tools.py
!python data/generate_dataset.py
!python eval_lion.py data data/train.jsonl data/val.jsonl


In [ ]:
#@title ٣-ب) توسيع البيانات: محادثة + إجابة مستندة إلى مصادر + ترشيح تكرار (≈١٥–٣٠ دقيقة)
# ١) أمثلة محادثة عربية بنموذج معلّم مفتوح (Qwen2.5-7B 4-bit) · ٢) أمثلة «إجابة من مصادر» بنفس صيغة التطبيق:
#    حتمية بلا شبكة + مقتطفات ويكيبيديا العربية الحقيقية · ٣) حذف التكرار وتسرّب val.
# كل خطوة في عملية منفصلة فتتحرر ذاكرة GPU. اجعلها False لتخطّيها (بيانات صغيرة = جودة ضعيفة).
EXPAND_DATA = True
N_EXAMPLES = 400   #@param {type:"integer"}
N_GROUNDED = 200   #@param {type:"integer"}
if EXPAND_DATA:
    !python distill_local.py --n {N_EXAMPLES}
    !python distill_grounded.py --offline
    !python distill_grounded.py --wiki --n {N_GROUNDED}
    !python data/generate_dataset.py
    !python data/dedupe.py
    !python eval_lion.py data data/train.jsonl data/val.jsonl
    !shuf -n 3 data/seed_extra.jsonl | cut -c1-400   # راجع العيّنة بعينك قبل التدريب


In [ ]:
#@title ٣-ج) التعلّم من مزوّديك عبر البوابة (اختياري — راجع شروط كل مزوّد أولاً)
# قبل التشغيل: 🔑 Add new secret باسم LION_API_KEY (توكن البوابة) وفعّل Notebook access.
# اعرف النماذج المتاحة أولاً بتشغيل: !python distill.py --url {GATEWAY_URL} --list-models
USE_PROVIDERS = False
GATEWAY_URL = "https://lionai-gateway.starsyria2500.workers.dev/v1"   #@param {type:"string"}
MODELS = ""        #@param {type:"string"}   # مثال: مزود/نموذج1,مزود/نموذج2,مزود/نموذج3
N_PROVIDER = 300   #@param {type:"integer"}
AGREE = True       #@param {type:"boolean"}   # اشترط اتفاق نموذج ثانٍ (يقلّل الهلوسة)
if USE_PROVIDERS:
    import os
    from google.colab import userdata
    os.environ["LION_API_KEY"] = userdata.get("LION_API_KEY")
    if not MODELS.strip():
        raise SystemExit("اكتب MODELS (شغّل --list-models لمعرفة المتاح).")
    agree = "--agree" if AGREE else ""
    !python distill.py --url {GATEWAY_URL} --models {MODELS} --auto {N_PROVIDER} {agree}
    !python data/generate_dataset.py
    !python data/dedupe.py
    !python eval_lion.py data data/train.jsonl data/val.jsonl
    # سجل مصدر كل مثال: data/distill_log.jsonl — احذف منه/من seed_extra ما يخالف شروط أي مزوّد.


## توسيع البيانات (اختياري لكنه الأهم للجودة)

١٨٠ مثالاً تكفي لتعلّم الشكل، لا للجودة العالية. الأفضل ٥٠٠–٢٠٠٠ مثال.
- أضف أمثلتك في `model/data/seed_extra.jsonl` (سطر JSON لكل مثال: `{"messages":[...]}`).
- أو ولّدها من مزوّد قوي عبر بوابتك:
  ```
  !LION_API_KEY=... python distill.py prompts.txt --url <gateway>/v1 --model <provider/model>
  ```
  راجع الأمثلة المولّدة يدوياً قبل التدريب، واحترم شروط استخدام المزوّد.

> الخلية **٣-ب** تولّد وترشّح مئات الأمثلة تلقائياً بنموذج معلّم مفتوح (بلا مفتاح API). تحقّق من العيّنة التي تطبعها.


In [ ]:
#@title ٤) تشغيل تجريبي سريع (١٢ خطوة — يكشف أي تعارض في إصدارات TRL)
!python train_lora.py --dry_run --base Qwen/Qwen2.5-1.5B-Instruct --out out/dry


In [ ]:
#@title ٥) التدريب الكامل (٤٥–٩٠ دقيقة)
# 1.5B على T4: ~45–60 دقيقة. للأجهزة 4GB رام أبقِ 1.5B.
# للنموذج الأصغر (أسرع لكن أضعف): --base Qwen/Qwen2.5-0.5B-Instruct
!python train_lora.py --base Qwen/Qwen2.5-1.5B-Instruct --epochs 3 --max_len 2048 --out out/lion-1-lora


In [ ]:
#@title ٦) دمج LoRA في النموذج الأساسي (٢–٣ دقائق)
!python merge_export.py --base Qwen/Qwen2.5-1.5B-Instruct --lora out/lion-1-lora --out out/lion-1-merged


In [ ]:
#@title ٦-ب) بوابة الجودة قبل النشر (٢–٥ دقائق)
# تختبر النموذج المدمج: هوية، عدم تسريب، سؤال معرفي بلا أداة، واختيار أدوات صحيح على val.
# إن قلّت الدرجة عن GATE لن تنشر الخلية التالية إلا بتجاوز صريح (FORCE_PUBLISH).
import subprocess
GATE = 0.6   #@param {type:"number"}
r = subprocess.run(["python", "eval_merged.py", "--merged", "out/lion-1-merged", "--gate", str(GATE)])
EVAL_OK = (r.returncode == 0)
print("EVAL_OK =", EVAL_OK)


In [ ]:
#@title ٧) احتياط: حفظ LoRA والنموذج على Google Drive (اختياري)
# الكولاب المجاني قد يُقطع بعد ساعات. هذه الخلية تحفظ محوّل LoRA والنموذج المدمج على درايفك.
SAVE_TO_DRIVE = True
if SAVE_TO_DRIVE:
    from google.colab import drive
    import shutil, os
    drive.mount('/content/drive')
    dest = '/content/drive/MyDrive/Lion-1'
    os.makedirs(dest, exist_ok=True)
    for src in ['out/lion-1-lora', 'out/lion-1-merged']:
        if os.path.isdir(src):
            target = os.path.join(dest, os.path.basename(src))
            shutil.rmtree(target, ignore_errors=True)
            shutil.copytree(src, target)
            print('✅ حُفظ:', target)


In [ ]:
#@title ٨) تحويل إلى GGUF كمّي Q4_K_M (١٠ دقائق أول مرة)
# أول تشغيل يثبّت متطلبات التحويل ويبني llama-quantize (~10 دقائق).
# Q4_K_M ≈ 940MB وجيد لهاتف 4GB · Q5_K_M أدق وأثقل · Q8_0 أثقل بكثير.
!python export_gguf.py --merged out/lion-1-merged --quant Q4_K_M --name lion-1


In [ ]:
#@title ٨-ب) النشر على Hugging Face (يرفع GGUF + بطاقة النموذج)
# المفتاح: إما Secrets (🔑 على الكمبيوتر) باسم HF_TOKEN، أو يطلبه منك الدفتر في حقل مخفي عند التشغيل (مناسب للجوال).
# لا تكتب المفتاح في الكود أبداً.
HF_REPO = ""          #@param {type:"string"}   # مثال: اسم_مستخدمك/lion-1
BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"   #@param {type:"string"}   # يجب أن يطابق ما دُرّب عليه
PRIVATE = False       #@param {type:"boolean"}
FORCE_PUBLISH = False #@param {type:"boolean"}   # تجاوز بوابة الجودة (غير مستحسن)
import glob, os, subprocess, sys
from google.colab import userdata
if not HF_REPO or "/" not in HF_REPO:
    raise SystemExit("اكتب HF_REPO بالشكل: اسم_مستخدمك/lion-1")
if not globals().get("EVAL_OK", False) and not FORCE_PUBLISH:
    raise SystemExit("لم تجتز بوابة الجودة (أو لم تُشغَّل خلية ٦-ب). شغّلها أو فعّل FORCE_PUBLISH عن علم.")
try:
    token = userdata.get("HF_TOKEN")          # الطريقة الأولى: Secrets (🔑) إن وُجدت
except Exception:
    import getpass                             # الطريقة الثانية (للجوال): حقل مخفي، لا يُحفظ في الدفتر
    token = getpass.getpass("الصق مفتاح Hugging Face (نوع Write) ثم Enter: ").strip()
if not token.startswith("hf_"):
    raise SystemExit("المفتاح يجب أن يبدأ بـ hf_ — أنشئه من huggingface.co/settings/tokens بنوع Write.")
gguf = sorted(glob.glob("/content/LionAI/model/out/*.gguf"), key=os.path.getsize)
if not gguf:
    raise SystemExit("لا يوجد ملف GGUF — شغّل خلية التحويل ٨) أولاً.")
cmd = [sys.executable, "publish_hf.py", "--gguf", gguf[-1], "--repo", HF_REPO, "--base", BASE_MODEL]
if PRIVATE:
    cmd.append("--private")
subprocess.run(cmd, env={**os.environ, "HF_TOKEN": token}, check=True)


In [ ]:
#@title ٩) تنزيل ملف GGUF النهائي إلى جهازك
import glob, os
from google.colab import files      # يعمل داخل Colab فقط
fs = sorted(glob.glob("/content/LionAI/model/out/*.gguf"), key=os.path.getsize)
for f in fs:
    print(f, round(os.path.getsize(f) / 1024 / 1024), "MB")
if fs:
    files.download(fs[-1])


## تشغيل النموذج داخل التطبيق

1. النشر على HuggingFace يتم بخلية ٨-ب أعلاه (توكن Write في Secrets باسم HF_TOKEN) — أو انسخ الملف إلى ذاكرة الهاتف.
2. التطبيق ← **البيئة المحلية** ← بطاقة **🦁 محرّك Lion-1** ← **＋ نموذج مخصص** ←
   الصق رابط `…/resolve/main/lion-1-Q4_K_M.gguf` أو مسار الملف ← **حفظ**.
3. **⬇️ تنزيل النموذج** ← **▶️ تشغيل** ← **🧪 اختبار**.
4. للجودة قبل النقل للهاتف: `python eval_lion.py live --url http://127.0.0.1:8089/v1 --model lion-1`
   بعد `./serve.sh` على حاسوبك.

## تحذيرات صادقة
- **الإفراط في التخصيص (overfit)**: مع ١٨٠ مثالاً و٣ دورات قد يحفظ الأجوبة. راقب `eval_loss`
  في السجل؛ إن ارتفع بينما `train_loss` ينخفض فأنقص الدورات إلى ٢ أو زد البيانات.
- **الحقوق**: درّب على بياناتك أو على بيانات مرخّصة. لا تستخدم مخرجات مزوّد بما يخالف شروطه.
- النموذج الناتج **يقلّد الأسلوب**، وقد يخطئ في المعرفة كما يخطئ الأساس — لا تعتمد عليه في وقائع بلا أداة تحقّق.
